## Carregando o conjunto de dados Iris


In [9]:
from sklearn.datasets import load_iris

iris = load_iris()

## Processamento dos dados

1. Normalizar (`StandardScaler`)
$$
normalizado_i = \frac{x_i - media(X)}{desvpadrao(X)}
$$
1. Dividir em treino e teste

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = iris.data
y = iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Construir a rede neural

Nesta rede neural com 3 classes, teremos 3 neurônios de saída. O operador Softmax aplica uma exponencial a cada uma dessas saídas (logits) e então as divide pela soma de todas as exponenciais.

O exponencial é necessário para tratar números positivos e negativos, já que:

$$ e^2 = 7,3891 $$
$$ e^{-2} = 0,1353 $$

Isso gera um vetor de probabilidades que somam 1, indicando o percentual de confiança da rede para cada classe.


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2)) # O dropout remove aleatoriamente 20% dos neurônios de
                        # uma camada (para uma época específica), evitando que
                        # um neurônio aprenda demais e anule os demais
model.add(Dense(32, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,499 (9.76 KB)

 Trainable params: 2,499 (9.76 KB)

 Non-trainable params: 0 (0.00 B)

## Configurando o Modelo Aprendizado


### Otimizador (ajuste do parâmetro de aprendizagem)

Adam usa uma abordagem adaptativa para ajustar a taxa de aprendizado (o tamanho do passo que damos em direção ao mínimo). Em vez de usar a mesma taxa de aprendizado para todos os parâmetros, ele ajusta a taxa de aprendizado para cada parâmetro individualmente.
  - Se o histórico do gradiente for sempre em um sentido ele aumenta a taxa de aprendizado
  - Se o histórico do gradiente ficar alternando muito (positivo e negativo) ele abaixa a taxa de aprendizado.
  - Ele funciona individualmente para cada conexão (peso) da rede.


### Loss: Sparse Categorical

É o equivalente ao que vimos o Binary Cross-Entropy só que para múltiplas classes. Em vez da fórmula que vimos:
$$
Loss = y \ln{\hat{y}} + (1-y)\ln{(1 - \hat{y}})
$$

Não usamos a parte do 1 - y, em vez disso teremos várias classes, então para cada classe (C) faremos:
$$
\sum_{i=1}^{C}{y\ln{\hat{y}}}
$$

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## Treinar o modelo


In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.3804 - loss: 1.1069 - val_accuracy: 0.5333 - val_loss: 0.9865
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5650 - loss: 0.9919 - val_accuracy: 0.6000 - val_loss: 0.8829
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6398 - loss: 0.8989 - val_accuracy: 0.8333 - val_loss: 0.7941
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6831 - loss: 0.8444 - val_accuracy: 0.9000 - val_loss: 0.7190
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7542 - loss: 0.7593 - val_accuracy: 0.9000 - val_loss: 0.6554
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7748 - loss: 0.6901 - val_accuracy: 0.9000 - val_loss: 0.6020
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8144 - loss: 0.6511 - val_accuracy: 0.8667 - val_loss: 0.5572
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8340 - loss: 0.5661 - val_accuracy: 0.8667 - val_loss: 0.5176


## Avaliar o modelo

In [ ]:
evaluation_results = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {evaluation_results[0]:.4f}")
print(f"Test Accuracy: {evaluation_results[1]:.4f}")

Test Loss: 0.1134
Test Accuracy: 1.0000
